#  Cervical Cancer Multi-Modal Deep Learning Classification


---

###  Datasets:
- **SIPaKMeD**: 4,049 images (University of Ioannina)
- **Herlev**: 917 images (MDE Lab - DTU/Herlev)
- **UCI Clinical**: Risk factors data
- **Genomic**: TCGA-CESC style simulation (realistic, no leakage)


###  Instructions:
1. **Runtime → Change runtime type → GPU (T4)**
2. **Runtime → Run all**
3. Wait ~60 minutes for completion

---

In [ ]:
#@title 1. Install Packages
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q timm==0.9.12 albumentations==1.3.1
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q tqdm pillow scipy opencv-python-headless
!apt-get install -qq p7zip-full
!pip install -q statsmodels
print(" All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 8.1 MB/s eta 0:00:00
 All packages installed!


In [ ]:
#@title 2. Import Libraries & Setup
import os
import sys
import warnings
import random
import json
import glob
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast
import timm

import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"� Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

for d in ['data', 'data/sipakmed', 'data/herlev', 'data/clinical', 'models', 'results', 'figures']:
    os.makedirs(d, exist_ok=True)

print("\n Environment ready!")

� Device: cuda
   GPU: Tesla T4

 Environment ready!


In [ ]:
#@title 3. Configuration
class Config:
    EXPERIMENT_NAME = "CervicalCancer_MultiModal__Fixed"
    SEED = 42
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_WORKERS = 4  # L4 has ample CPU
    NUM_EPOCHS = 30
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-4
    EARLY_STOPPING_PATIENCE = 10
    NUM_CLASSES = 2
    DROPOUT_RATE = 0.4
    MC_DROPOUT_SAMPLES = 20
    MODALITY_DROPOUT_PROB = 0.25
    USE_AMP = True
    COLORS = {
        'primary': '#0077BB', 'secondary': '#EE7733', 'success': '#009988',
        'warning': '#EE3377', 'danger': '#CC3311',
        'image': '#0077BB', 'clinical': '#EE7733', 'genomic': '#009988', 'fusion': '#EE3377'
    }

config = Config()
print(f" Experiment: {config.EXPERIMENT_NAME}")

 Experiment: CervicalCancer_MultiModal__Fixed


In [ ]:
#@title 4. � Download SIPaKMeD Dataset (Official Source)
print("="*70)
print("� DOWNLOADING SIPaKMeD DATASET")
print("   Source: University of Ioannina, Greece")
print("   URL: https://www.cs.uoi.gr/~marina/sipakmed.html")
print("="*70)

SIPAKMED_BASE = "https://www.cs.uoi.gr/~marina/SIPAKMED"
SIPAKMED_FILES = [
    ("im_Superficial-Intermediate.7z", "Superficial-Intermediate", 0),
    ("im_Parabasal.7z", "Parabasal", 0),
    ("im_Metaplastic.7z", "Metaplastic", 0),
    ("im_Koilocytotic.7z", "Koilocytotic", 1),
    ("im_Dyskeratotic.7z", "Dyskeratotic", 1),
]

sipakmed_paths = []
sipakmed_labels = []

for filename, classname, label in SIPAKMED_FILES:
    url = f"{SIPAKMED_BASE}/{filename}"
    local_7z = f"data/sipakmed/{filename}"
    extract_dir = f"data/sipakmed/{classname}"

    existing_images = glob.glob(f"{extract_dir}/**/*.bmp", recursive=True) + \
                      glob.glob(f"{extract_dir}/**/*.BMP", recursive=True)

    if len(existing_images) > 0:
        print(f"    {classname}: {len(existing_images)} images (already exists)")
    else:
        print(f"   � Downloading {classname}...")
        try:
            urllib.request.urlretrieve(url, local_7z)
            print(f"      Downloaded: {os.path.getsize(local_7z)/1e6:.1f} MB")
            os.makedirs(extract_dir, exist_ok=True)
            result = subprocess.run(["7z", "x", "-y", f"-o{extract_dir}", local_7z],
                                    capture_output=True, text=True)
            if result.returncode == 0:
                print(f"       Extracted")
                os.remove(local_7z)
            else:
                print(f"       Extraction issue")
        except Exception as e:
            print(f"       Error: {e}")

    for ext in ['*.bmp', '*.BMP', '*.png', '*.jpg']:
        for img_path in glob.glob(f"{extract_dir}/**/{ext}", recursive=True):
            sipakmed_paths.append(img_path)
            sipakmed_labels.append(label)

print(f"\n SIPaKMeD Total: {len(sipakmed_paths)} images")
if len(sipakmed_paths) > 0:
    print(f"   Normal (0): {sipakmed_labels.count(0)}")
    print(f"   Abnormal (1): {sipakmed_labels.count(1)}")

� DOWNLOADING SIPaKMeD DATASET
   Source: University of Ioannina, Greece
   URL: https://www.cs.uoi.gr/~marina/sipakmed.html
   � Downloading Superficial-Intermediate...
      Downloaded: 762.9 MB
       Extracted
   � Downloading Parabasal...
      Downloaded: 548.7 MB
       Extracted
   � Downloading Metaplastic...
      Downloaded: 1396.5 MB
       Extracted
   � Downloading Koilocytotic...
      Downloaded: 1290.5 MB
       Extracted
   � Downloading Dyskeratotic...
      Downloaded: 1183.0 MB
       Extracted

 SIPaKMeD Total: 5015 images
   Normal (0): 2916
   Abnormal (1): 2099


In [ ]:
#@title 5. � Download Herlev Dataset (Official Source) - FIXED PATH DETECTION
print("\n" + "="*70)
print("� DOWNLOADING HERLEV DATASET")
print("   Source: MDE Lab, University of the Aegean / DTU")
print("="*70)

HERLEV_URL = "http://mde-lab.aegean.gr/images/stories/docs/smear2005.zip"
local_zip = "data/herlev/smear2005.zip"

# Check if already extracted - FIXED: search all subdirectories
existing_herlev = []
for ext in ['*.bmp', '*.BMP', '*.png', '*.jpg', '*.JPG']:
    existing_herlev.extend(glob.glob(f"data/herlev/**/{ext}", recursive=True))

if len(existing_herlev) > 100:
    print(f"    Already exists: {len(existing_herlev)} images")
else:
    print("   � Downloading smear2005.zip (85 MB)...")
    try:
        # Try wget first (more reliable)
        result = subprocess.run(
            ["wget", "-q", "--show-progress", "-O", local_zip, HERLEV_URL],
            capture_output=False
        )
        if not os.path.exists(local_zip) or os.path.getsize(local_zip) < 1000000:
            urllib.request.urlretrieve(HERLEV_URL, local_zip)
        print(f"      Downloaded: {os.path.getsize(local_zip)/1e6:.1f} MB")
    except Exception as e:
        print(f"      Trying urllib: {e}")
        urllib.request.urlretrieve(HERLEV_URL, local_zip)

    # Extract
    print("   � Extracting...")
    try:
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall("data/herlev")
        print("       Extracted!")
    except Exception as e:
        print(f"      Trying unzip command: {e}")
        subprocess.run(["unzip", "-o", "-q", local_zip, "-d", "data/herlev"])

# FIXED: Comprehensive path detection for Herlev
# Herlev has 7 classes in folders, we map to binary
HERLEV_CLASS_MAP = {
    # Normal classes (0)
    'superficial': 0, 'intermediate': 0, 'columnar': 0,
    'normal': 0, 'superficiel': 0,
    # Abnormal classes (1)
    'mild': 1, 'moderate': 1, 'severe': 1, 'carcinoma': 1,
    'dysplasia': 1, 'light': 1, 'cancer': 1, 'situ': 1
}

herlev_paths = []
herlev_labels = []

# Search all possible image locations
print("\n   � Scanning for Herlev images...")
all_herlev_images = []
for ext in ['*.bmp', '*.BMP', '*.png', '*.jpg', '*.JPG', '*.jpeg']:
    all_herlev_images.extend(glob.glob(f"data/herlev/**/{ext}", recursive=True))

print(f"      Found {len(all_herlev_images)} total image files")

# Classify each image based on path
for img_path in all_herlev_images:
    path_lower = img_path.lower()
    label = None

    for keyword, lbl in HERLEV_CLASS_MAP.items():
        if keyword in path_lower:
            label = lbl
            break

    if label is not None:
        herlev_paths.append(img_path)
        herlev_labels.append(label)

print(f"\n Herlev Total: {len(herlev_paths)} images")
if len(herlev_paths) > 0:
    print(f"   Normal (0): {herlev_labels.count(0)}")
    print(f"   Abnormal (1): {herlev_labels.count(1)}")
else:
    # Debug: show directory structure
    print("    No labeled images found. Directory structure:")
    for root, dirs, files in os.walk("data/herlev"):
        level = root.replace("data/herlev", "").count(os.sep)
        indent = "   " * (level + 1)
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:
            for f in files[:3]:
                print(f"{indent}  {f}")
            if len(files) > 3:
                print(f"{indent}  ... and {len(files)-3} more")


� DOWNLOADING HERLEV DATASET
   Source: MDE Lab, University of the Aegean / DTU
   � Downloading smear2005.zip (85 MB)...
      Downloaded: 89.3 MB
   � Extracting...
       Extracted!

   � Scanning for Herlev images...
      Found 1834 total image files

 Herlev Total: 1834 images
   Normal (0): 484
   Abnormal (1): 1350


In [ ]:
#@title 6. � Download UCI Clinical Data
print("\n" + "="*70)
print("� DOWNLOADING UCI CLINICAL DATA")
print("="*70)

clinical_path = "data/clinical/cervical_cancer.csv"

if os.path.exists(clinical_path):
    print("    Already exists!")
else:
    UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00383/risk_factors_cervical_cancer.csv"
    try:
        urllib.request.urlretrieve(UCI_URL, clinical_path)
        print("    Downloaded!")
    except Exception as e:
        print(f"    UCI download failed: {e}")

if os.path.exists(clinical_path):
    clinical_df = pd.read_csv(clinical_path)
    print(f"    Shape: {clinical_df.shape}")


� DOWNLOADING UCI CLINICAL DATA
    Downloaded!
    Shape: (858, 36)


In [ ]:
#@title 7. Combine All Image Data
print("\n" + "="*70)
print(" COMBINING ALL IMAGE DATASETS")
print("="*70)

all_paths = sipakmed_paths + herlev_paths
all_labels = sipakmed_labels + herlev_labels

# Track source dataset for per-dataset evaluation (REVISION: Reviewer 1.2g)
all_source = (['sipakmed'] * len(sipakmed_paths)) + (['herlev'] * len(herlev_paths))

# If insufficient data, create minimal synthetic supplement
if len(all_paths) < 500:
    print("\n Insufficient real data. Creating synthetic supplement...")

    def create_synthetic_cells(output_dir, n_per_class=300):
        classes = {
            'NILM': {'color': [0.75, 0.85, 0.92], 'nucleus_ratio': 0.18, 'label': 0},
            'LSIL': {'color': [0.82, 0.62, 0.72], 'nucleus_ratio': 0.38, 'label': 1},
            'HSIL': {'color': [0.88, 0.52, 0.58], 'nucleus_ratio': 0.48, 'label': 1},
        }
        paths, labels = [], []

        for cls_name, props in tqdm(classes.items(), desc="Creating synthetic"):
            cls_dir = os.path.join(output_dir, cls_name)
            os.makedirs(cls_dir, exist_ok=True)

            for i in range(n_per_class):
                img = np.ones((224, 224, 3), dtype=np.float32) * 0.95
                cx, cy = 112 + np.random.randint(-20, 20), 112 + np.random.randint(-20, 20)
                r = np.random.randint(45, 70)
                y, x = np.ogrid[:224, :224]
                cell_mask = (x - cx)**2 + (y - cy)**2 <= r**2
                img[cell_mask] = np.clip(np.array(props['color']) + np.random.uniform(-0.08, 0.08, 3), 0, 1)
                nr = int(r * props['nucleus_ratio'])
                nucleus_mask = (x - cx)**2 + (y - cy)**2 <= nr**2
                img[nucleus_mask] = [0.28, 0.22, 0.38] if props['label'] == 1 else [0.38, 0.32, 0.48]

                img_uint8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
                path = f"{cls_dir}/{cls_name}_{i:04d}.png"
                cv2.imwrite(path, cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR))
                paths.append(path)
                labels.append(props['label'])

        return paths, labels

    syn_paths, syn_labels = create_synthetic_cells("data/synthetic", n_per_class=400)
    all_paths.extend(syn_paths)
    all_labels.extend(syn_labels)
    all_source.extend(['synthetic'] * len(syn_paths))

all_labels = np.array(all_labels)
all_source = np.array(all_source)

print(f"\n" + "="*50)
print(f" FINAL IMAGE DATASET SUMMARY")
print(f"="*50)
print(f"   SIPaKMeD:  {len(sipakmed_paths)} images")
print(f"   Herlev:    {len(herlev_paths)} images")
print(f"   " + "-"*30)
print(f"   TOTAL:     {len(all_paths)} images")
print(f"\n   Class distribution:")
print(f"   Normal (0):   {sum(all_labels==0)} ({100*sum(all_labels==0)/len(all_labels):.1f}%)")
print(f"   Abnormal (1): {sum(all_labels==1)} ({100*sum(all_labels==1)/len(all_labels):.1f}%)")


 COMBINING ALL IMAGE DATASETS

 FINAL IMAGE DATASET SUMMARY
   SIPaKMeD:  5015 images
   Herlev:    1834 images
   ------------------------------
   TOTAL:     6849 images

   Class distribution:
   Normal (0):   3400 (49.6%)
   Abnormal (1): 3449 (50.4%)


In [ ]:
#@title 8. Load Clinical Data

def load_clinical_data(n_samples):
    """Load UCI clinical data or create realistic simulation."""
    clinical_path = "data/clinical/cervical_cancer.csv"

    if os.path.exists(clinical_path):
        print("� Loading UCI clinical data...")
        df = pd.read_csv(clinical_path)
        df = df.replace('?', np.nan)

        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        target_cols = ['Hinselmann', 'Schiller', 'Citology', 'Biopsy']
        feature_cols = [c for c in df.columns if c not in target_cols]

        X = df[feature_cols].values
        imputer = SimpleImputer(strategy='median')
        X = imputer.fit_transform(X)
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        # Resample to match image count
        idx = np.random.choice(len(X), n_samples, replace=True)
        X = X[idx]

        print(f"   Shape: {X.shape}")
        return X, scaler, feature_cols
    else:
        print("� Creating simulated clinical data...")
        np.random.seed(42)
        feature_names = ['Age', 'NumPartners', 'FirstIntercourse', 'NumPregnancies',
                         'Smokes', 'SmokesYears', 'HormonalContraceptives',
                         'HormonalYears', 'IUD', 'IUDYears', 'STDs', 'STDsNum']
        features = np.column_stack([
            np.random.normal(38, 12, n_samples).clip(18, 80),
            np.random.poisson(3, n_samples).clip(0, 20),
            np.random.normal(17, 3, n_samples).clip(12, 35),
            np.random.poisson(2, n_samples).clip(0, 15),
            np.random.choice([0, 1], n_samples, p=[0.7, 0.3]),
            np.random.exponential(5, n_samples).clip(0, 40),
            np.random.choice([0, 1], n_samples, p=[0.5, 0.5]),
            np.random.exponential(4, n_samples).clip(0, 30),
            np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
            np.random.exponential(3, n_samples).clip(0, 20),
            np.random.choice([0, 1], n_samples, p=[0.85, 0.15]),
            np.random.poisson(0.3, n_samples).clip(0, 5),
        ])
        scaler = StandardScaler()
        X = scaler.fit_transform(features)
        print(f"   Shape: {X.shape}")
        return X, scaler, feature_names

clinical_X, clinical_scaler, clinical_features = load_clinical_data(len(all_labels))
CLINICAL_DIM = clinical_X.shape[1]
print(f"   Clinical features: {CLINICAL_DIM}")

� Loading UCI clinical data...
   Shape: (6849, 32)
   Clinical features: 32


In [ ]:
#@title 9. Create Genomic Features - FIXED (No Data Leakage)

def create_genomic_features_realistic(n_samples, labels, noise_level=0.7):
    """
    Create REALISTIC genomic features WITHOUT data leakage.

    Key fixes:
    1. Features are generated INDEPENDENTLY of labels first
    2. Only weak statistical association is added (not deterministic)
    3. High noise ensures no perfect separation
    4. Mimics real TCGA data characteristics
    """
    print("� Creating REALISTIC genomic features (no leakage)...")
    np.random.seed(42)
    labels = np.array(labels)

    N_GENES = 80    # Gene expression features
    N_MUT = 20      # Mutation features
    N_METH = 40     # Methylation features

    # === GENE EXPRESSION ===
    # Base expression: log-normal distribution (realistic for RNA-seq)
    gene_expr = np.random.lognormal(mean=4, sigma=1.5, size=(n_samples, N_GENES))

    # Add WEAK label-associated signal to only 5 genes (realistic)
    # Effect size is small and noisy - won't allow perfect classification
    for i in range(5):
        base_effect = np.random.uniform(0.1, 0.3)  # Small effect
        noise = np.random.normal(0, 0.5, n_samples)  # Large noise
        gene_expr[:, i] += labels * base_effect + noise

    # Add batch effects and technical noise (realistic)
    batch_effect = np.random.normal(0, 0.3, (1, N_GENES))
    gene_expr += batch_effect
    gene_expr += np.random.normal(0, noise_level, gene_expr.shape)

    # === MUTATIONS ===
    # Binary mutation matrix - mostly zeros (realistic)
    mutation_rate = 0.05  # 5% base mutation rate
    mutations = np.random.binomial(1, mutation_rate, (n_samples, N_MUT)).astype(float)

    # Only 2-3 mutations have weak association with cancer
    # NOT deterministic - just slightly higher probability
    for i in range(3):
        prob_if_cancer = 0.12  # 12% if cancer
        prob_if_normal = 0.04  # 4% if normal
        probs = np.where(labels == 1, prob_if_cancer, prob_if_normal)
        mutations[:, i] = np.random.binomial(1, probs)

    # === METHYLATION ===
    # Beta values between 0-1 (realistic for methylation)
    methylation = np.random.beta(2, 5, (n_samples, N_METH))

    # Weak hypermethylation signal in 5 sites for cancer
    for i in range(5):
        shift = labels * np.random.uniform(0.05, 0.15)  # Small shift
        noise = np.random.normal(0, 0.1, n_samples)
        methylation[:, i] = np.clip(methylation[:, i] + shift + noise, 0, 1)

    # === COMBINE AND NORMALIZE ===
    genomic = np.hstack([gene_expr, mutations, methylation])

    # Add global noise to prevent any perfect separation
    genomic += np.random.normal(0, noise_level * 0.5, genomic.shape)

    # Standardize
    scaler = StandardScaler()
    genomic = scaler.fit_transform(genomic)

    # Verify no leakage: compute simple correlation
    correlations = [np.abs(np.corrcoef(genomic[:, i], labels)[0, 1]) for i in range(genomic.shape[1])]
    max_corr = np.max(correlations)
    mean_corr = np.mean(correlations)

    print(f"   Shape: {genomic.shape}")
    print(f"   Max feature-label correlation: {max_corr:.3f} (should be < 0.3)")
    print(f"   Mean feature-label correlation: {mean_corr:.3f} (should be < 0.1)")

    if max_corr > 0.5:
        print("    Warning: High correlation detected, adding more noise...")
        genomic += np.random.normal(0, 0.5, genomic.shape)
        genomic = StandardScaler().fit_transform(genomic)

    return genomic, scaler

genomic_X, genomic_scaler = create_genomic_features_realistic(len(all_labels), all_labels, noise_level=0.7)
GENOMIC_DIM = genomic_X.shape[1]
print(f"   Genomic features: {GENOMIC_DIM}")

� Creating REALISTIC genomic features (no leakage)...
   Shape: (6849, 140)
   Max feature-label correlation: 0.138 (should be < 0.3)
   Mean feature-label correlation: 0.014 (should be < 0.1)
   Genomic features: 140


In [ ]:
#@title 10. Dataset & DataLoaders

class MultiModalDataset(Dataset):
    def __init__(self, image_paths, clinical, genomic, labels, transform=None, modality_dropout=0.0, training=True):
        self.image_paths = image_paths
        self.clinical = clinical
        self.genomic = genomic
        self.labels = labels
        self.transform = transform
        self.modality_dropout = modality_dropout
        self.training = training

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        try:
            img = cv2.imread(self.image_paths[idx])
            if img is None:
                img = np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        except:
            img = np.zeros((224, 224, 3), dtype=np.uint8)

        if self.transform:
            img = self.transform(image=img)['image']
        else:
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        clinical = torch.tensor(self.clinical[idx], dtype=torch.float32)
        genomic = torch.tensor(self.genomic[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        mask = torch.ones(3)
        if self.training and self.modality_dropout > 0:
            drop = torch.rand(3) > self.modality_dropout
            if drop.sum() == 0:
                drop[torch.randint(0, 3, (1,))] = True
            mask = drop.float()

        return {'image': img, 'clinical': clinical, 'genomic': genomic, 'label': label, 'modality_mask': mask}

# Transforms
train_transform = A.Compose([
    A.Resize(config.IMAGE_SIZE, config.IMAGE_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=20, p=0.5),
    A.OneOf([A.GaussNoise(var_limit=(10, 50)), A.GaussianBlur(), A.MotionBlur()], p=0.3),
    A.OneOf([A.RandomBrightnessContrast(), A.HueSaturationValue()], p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(config.IMAGE_SIZE, config.IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Stratified split
indices = np.arange(len(all_labels))
train_val_idx, test_idx = train_test_split(indices, test_size=0.15, stratify=all_labels, random_state=42)
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.18, stratify=all_labels[train_val_idx], random_state=42)

# REVISION: keep test source labels aligned to test_idx order for per-dataset eval
test_source = all_source[test_idx]
np.save("results/test_idx.npy", test_idx)  # reproducibility: exact split indices

print(f" Data Split:")
print(f"   Train: {len(train_idx)} samples")
print(f"   Val:   {len(val_idx)} samples")
print(f"   Test:  {len(test_idx)} samples")

# Create datasets
train_dataset = MultiModalDataset(
    [all_paths[i] for i in train_idx], clinical_X[train_idx], genomic_X[train_idx],
    all_labels[train_idx], train_transform, config.MODALITY_DROPOUT_PROB, True
)
val_dataset = MultiModalDataset(
    [all_paths[i] for i in val_idx], clinical_X[val_idx], genomic_X[val_idx],
    all_labels[val_idx], val_transform, 0.0, False
)
test_dataset = MultiModalDataset(
    [all_paths[i] for i in test_idx], clinical_X[test_idx], genomic_X[test_idx],
    all_labels[test_idx], val_transform, 0.0, False
)

# Weighted sampler for class imbalance
train_labels = all_labels[train_idx]
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, sampler=sampler,
                          num_workers=config.NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                         num_workers=config.NUM_WORKERS, pin_memory=True)

print("\n DataLoaders ready!")

 Data Split:
   Train: 4773 samples
   Val:   1048 samples
   Test:  1028 samples

 DataLoaders ready!


In [ ]:
#@title 11. Model Architecture

class ImageEncoder(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        self.resnet = timm.create_model('resnet50', pretrained=True, num_classes=0)
        self.densenet = timm.create_model('densenet121', pretrained=True, num_classes=0)
        self.efficientnet = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)

        total_dim = self.resnet.num_features + self.densenet.num_features + self.efficientnet.num_features

        self.fusion = nn.Sequential(
            nn.Linear(total_dim, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout)
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x, return_features=False):
        f1 = self.resnet(x)
        f2 = self.densenet(x)
        f3 = self.efficientnet(x)
        features = self.fusion(torch.cat([f1, f2, f3], dim=1))
        logits = self.classifier(features)
        return (logits, features) if return_features else logits

class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim, num_classes=2, dropout=0.4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.GELU(), nn.Dropout(dropout)
        )
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x, return_features=False):
        features = self.encoder(x)
        logits = self.classifier(features)
        return (logits, features) if return_features else logits

class GenomicEncoder(nn.Module):
    def __init__(self, input_dim, num_classes=2, dropout=0.4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(dropout)
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x, return_features=False):
        features = self.encoder(x)
        logits = self.classifier(features)
        return (logits, features) if return_features else logits

class MCDropoutWrapper(nn.Module):
    def __init__(self, model, n_samples=20):
        super().__init__()
        self.model = model
        self.n_samples = n_samples

    def enable_dropout(self):
        for m in self.model.modules():
            if isinstance(m, nn.Dropout):
                m.train()

    def forward(self, x, return_uncertainty=False, return_features=False):
        if not return_uncertainty:
            return self.model(x, return_features=return_features)

        self.enable_dropout()
        preds = []
        with torch.no_grad():
            for _ in range(self.n_samples):
                out = self.model(x)
                preds.append(F.softmax(out, dim=-1))

        preds = torch.stack(preds, dim=0)
        mean_pred = preds.mean(dim=0)
        uncertainty = -(mean_pred * torch.log(mean_pred + 1e-10)).sum(dim=-1) / np.log(mean_pred.shape[-1])
        return mean_pred, uncertainty

class UncertaintyAwareFusion(nn.Module):
    def __init__(self, n_modalities=3, num_classes=2):
        super().__init__()
        self.base_weights = nn.Parameter(torch.ones(n_modalities) / n_modalities)
        self.temperature = nn.Parameter(torch.tensor(1.0))

    def forward(self, predictions, uncertainties, modality_mask):
        certainty = 1.0 - uncertainties.clamp(0, 1)
        weighted = certainty * F.softplus(self.base_weights).unsqueeze(0) * modality_mask
        fusion_weights = F.softmax(weighted / (self.temperature.abs() + 0.1), dim=1)
        fused = (predictions * fusion_weights.unsqueeze(-1)).sum(dim=1)
        fused = fused / (fused.sum(dim=-1, keepdim=True) + 1e-10)
        confidence = (certainty * fusion_weights).sum(dim=1)
        return fused, fusion_weights, confidence

class MultiModalModel(nn.Module):
    def __init__(self, clinical_dim, genomic_dim, num_classes=2, dropout=0.4, mc_samples=20):
        super().__init__()
        self.image_encoder = ImageEncoder(num_classes, dropout)
        self.clinical_encoder = ClinicalEncoder(clinical_dim, num_classes, dropout)
        self.genomic_encoder = GenomicEncoder(genomic_dim, num_classes, dropout)

        self.image_mc = MCDropoutWrapper(self.image_encoder, mc_samples)
        self.clinical_mc = MCDropoutWrapper(self.clinical_encoder, mc_samples)
        self.genomic_mc = MCDropoutWrapper(self.genomic_encoder, mc_samples)

        self.fusion = UncertaintyAwareFusion(3, num_classes)

    def forward(self, image, clinical, genomic, mask, return_uncertainty=False, return_details=False):
        B = image.shape[0]

        if return_uncertainty:
            img_pred, img_unc = self.image_mc(image, return_uncertainty=True)
            clin_pred, clin_unc = self.clinical_mc(clinical, return_uncertainty=True)
            gen_pred, gen_unc = self.genomic_mc(genomic, return_uncertainty=True)
            preds = torch.stack([img_pred, clin_pred, gen_pred], dim=1)
            uncs = torch.stack([img_unc, clin_unc, gen_unc], dim=1)
        else:
            img_pred = F.softmax(self.image_encoder(image), dim=-1)
            clin_pred = F.softmax(self.clinical_encoder(clinical), dim=-1)
            gen_pred = F.softmax(self.genomic_encoder(genomic), dim=-1)
            preds = torch.stack([img_pred, clin_pred, gen_pred], dim=1)
            uncs = torch.ones(B, 3, device=image.device) * 0.1

        preds = preds * mask.unsqueeze(-1)
        fused, weights, conf = self.fusion(preds, uncs, mask)

        if return_details:
            return {
                'prediction': fused, 'fusion_weights': weights, 'confidence': conf,
                'modality_preds': preds, 'modality_uncs': uncs
            }
        return fused, conf

# Create model
model = MultiModalModel(CLINICAL_DIM, GENOMIC_DIM, config.NUM_CLASSES,
                        config.DROPOUT_RATE, config.MC_DROPOUT_SAMPLES).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n Model Summary:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            


 Model Summary:
   Total parameters: 36,924,902
   Trainable parameters: 36,924,902


In [ ]:
#@title 12. Training Functions

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce
        return focal_loss.mean()

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_state = None

    def __call__(self, score, model):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

class MetricTracker:
    def __init__(self):
        self.history = defaultdict(list)

    def update(self, **kwargs):
        for k, v in kwargs.items():
            self.history[k].append(v)

    def get(self, key):
        return self.history[key]

def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="Training", leave=False):
        img = batch['image'].to(device)
        clin = batch['clinical'].to(device)
        gen = batch['genomic'].to(device)
        labels = batch['label'].to(device)
        mask = batch['modality_mask'].to(device)

        optimizer.zero_grad()

        with autocast(enabled=config.USE_AMP):
            pred, _ = model(img, clin, gen, mask)
            loss = criterion(pred, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        all_preds.extend(pred.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), accuracy_score(all_labels, all_preds)

@torch.no_grad()
def evaluate(model, loader, criterion, device, return_preds=False):
    model.eval()
    total_loss = 0
    all_preds, all_probs, all_labels = [], [], []
    all_confs, all_weights = [], []
    all_mod_preds, all_mod_uncs = [], []

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        img = batch['image'].to(device)
        clin = batch['clinical'].to(device)
        gen = batch['genomic'].to(device)
        labels = batch['label'].to(device)
        mask = batch['modality_mask'].to(device)

        out = model(img, clin, gen, mask, return_uncertainty=True, return_details=True)
        pred = out['prediction']

        loss = criterion(pred, labels)
        total_loss += loss.item()

        all_probs.extend(pred.cpu().numpy())
        all_preds.extend(pred.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_confs.extend(out['confidence'].cpu().numpy())
        all_weights.extend(out['fusion_weights'].cpu().numpy())
        all_mod_preds.extend(out['modality_preds'].cpu().numpy())
        all_mod_uncs.extend(out['modality_uncs'].cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    metrics = {
        'loss': total_loss / len(loader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'auc': roc_auc_score(all_labels, all_probs[:, 1]) if len(np.unique(all_labels)) > 1 else 0.5,
        'confidence': np.mean(all_confs)
    }

    if return_preds:
        return metrics, {
            'labels': all_labels, 'predictions': all_preds, 'probabilities': all_probs,
            'confidences': np.array(all_confs), 'fusion_weights': np.array(all_weights),
            'modality_preds': np.array(all_mod_preds), 'modality_uncs': np.array(all_mod_uncs)
        }
    return metrics

print(" Training functions ready!")

 Training functions ready!


In [ ]:
#@title 13. 🏋 Train Model

criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
total_steps = len(train_loader) * config.NUM_EPOCHS
scheduler = OneCycleLR(optimizer, max_lr=config.LEARNING_RATE, total_steps=total_steps, pct_start=0.1)
scaler = GradScaler(enabled=config.USE_AMP)
early_stopping = EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE)
tracker = MetricTracker()

print("\n" + "="*60)
print("🏋 STARTING TRAINING")
print("="*60)

best_auc = 0
for epoch in range(config.NUM_EPOCHS):
    print(f"\n� Epoch {epoch+1}/{config.NUM_EPOCHS}")

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, device)
    val_metrics = evaluate(model, val_loader, criterion, device)

    tracker.update(
        train_loss=train_loss, train_acc=train_acc,
        val_loss=val_metrics['loss'], val_acc=val_metrics['accuracy'],
        val_auc=val_metrics['auc'], val_f1=val_metrics['f1'],
        lr=optimizer.param_groups[0]['lr']
    )

    print(f"   Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"   Val   - Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.4f} | AUC: {val_metrics['auc']:.4f}")

    if val_metrics['auc'] > best_auc:
        best_auc = val_metrics['auc']
        torch.save(model.state_dict(), 'models/best_model.pth')
        print(f"   � Best model saved! AUC: {best_auc:.4f}")

    early_stopping(val_metrics['auc'], model)
    if early_stopping.early_stop:
        print(f"\n Early stopping triggered at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(early_stopping.best_state)
model.to(device)

print(f"\n" + "="*60)
print(f" Training Complete!")
print(f"   Best Validation AUC: {best_auc:.4f}")
print("="*60)


🏋 STARTING TRAINING

� Epoch 1/30


Training:   0%|          | 0/150 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

   Train - Loss: 0.0400 | Acc: 0.6430
   Val   - Loss: 0.0280 | Acc: 0.8884 | AUC: 0.9513
   � Best model saved! AUC: 0.9513

� Epoch 2/30


Training:   0%|          | 0/150 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

   Train - Loss: 0.0164 | Acc: 0.8917
   Val   - Loss: 0.0095 | Acc: 0.9685 | AUC: 0.9968
   � Best model saved! AUC: 0.9968

� Epoch 30/30


Training:   0%|          | 0/150 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

   Train - Loss: 0.0163 | Acc: 0.8892
   Val   - Loss: 0.0093 | Acc: 0.9714 | AUC: 0.9964

 Training Complete!
   Best Validation AUC: 0.9968


In [ ]:
#@title 14.  Final Test Evaluation

final_metrics, pred_data = evaluate(model, test_loader, criterion, device, return_preds=True)

print("\n" + "="*60)
print(" FINAL TEST RESULTS")
print("="*60)
print(f"   Accuracy:   {final_metrics['accuracy']:.4f}")
print(f"   Precision:  {final_metrics['precision']:.4f}")
print(f"   Recall:     {final_metrics['recall']:.4f}")
print(f"   F1-Score:   {final_metrics['f1']:.4f}")
print(f"   AUC-ROC:    {final_metrics['auc']:.4f}")
print(f"   Confidence: {final_metrics['confidence']:.4f}")

print(f"\n{classification_report(pred_data['labels'], pred_data['predictions'], target_names=['Normal', 'Abnormal'], digits=4)}")

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]


 FINAL TEST RESULTS
   Accuracy:   0.9708
   Precision:  0.9621
   Recall:     0.9807
   F1-Score:   0.9713
   AUC-ROC:    0.9931
   Confidence: 0.9228

              precision    recall  f1-score   support

      Normal     0.9800    0.9608    0.9703       510
    Abnormal     0.9621    0.9807    0.9713       518

    accuracy                         0.9708      1028
   macro avg     0.9711    0.9707    0.9708      1028
weighted avg     0.9710    0.9708    0.9708      1028



In [ ]:
#@title 15. Bootstrap Confidence Intervals & Ablation Study

def bootstrap_ci(y_true, y_pred, y_prob, n_bootstrap=1000, ci=0.95):
    """Calculate bootstrap confidence intervals."""
    n = len(y_true)
    results = defaultdict(list)

    for _ in tqdm(range(n_bootstrap), desc="Bootstrap"):
        idx = np.random.choice(n, n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        results['accuracy'].append(accuracy_score(y_true[idx], y_pred[idx]))
        results['f1'].append(f1_score(y_true[idx], y_pred[idx], zero_division=0))
        results['auc'].append(roc_auc_score(y_true[idx], y_prob[idx][:, 1]))

    alpha = (1 - ci) / 2
    ci_results = {}
    for metric, values in results.items():
        values = np.array(values)
        ci_results[metric] = {
            'mean': np.mean(values),
            'ci_low': np.percentile(values, alpha * 100),
            'ci_high': np.percentile(values, (1 - alpha) * 100)
        }
    return ci_results

print("\n Calculating Bootstrap Confidence Intervals...")
ci_results = bootstrap_ci(pred_data['labels'], pred_data['predictions'], pred_data['probabilities'])

print("\n" + "="*60)
print(" 95% CONFIDENCE INTERVALS")
print("="*60)
for metric, values in ci_results.items():
    print(f"   {metric.upper():12s}: {values['mean']:.4f} ({values['ci_low']:.4f} - {values['ci_high']:.4f})")

# === ABLATION STUDY ===
print("\n" + "="*60)
print(" ABLATION STUDY")
print("="*60)

def evaluate_with_modality_mask(model, loader, device, mask_config):
    """Evaluate model with specific modality configuration."""
    model.eval()
    fixed_mask = torch.tensor([mask_config['image'], mask_config['clinical'], mask_config['genomic']]).float()

    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for batch in loader:
            img = batch['image'].to(device)
            clin = batch['clinical'].to(device)
            gen = batch['genomic'].to(device)
            labels = batch['label'].to(device)

            mask = fixed_mask.unsqueeze(0).expand(img.size(0), -1).to(device)
            pred, _ = model(img, clin, gen, mask)

            all_probs.extend(pred.cpu().numpy())
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    return {
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'auc': roc_auc_score(all_labels, all_probs[:, 1]) if len(np.unique(all_labels)) > 1 else 0.5
    }

ablation_configs = {
    'All Modalities': {'image': 1, 'clinical': 1, 'genomic': 1},
    'Image Only': {'image': 1, 'clinical': 0, 'genomic': 0},
    'Clinical Only': {'image': 0, 'clinical': 1, 'genomic': 0},
    'Genomic Only': {'image': 0, 'clinical': 0, 'genomic': 1},
    'Image+Clinical': {'image': 1, 'clinical': 1, 'genomic': 0},
    'Image+Genomic': {'image': 1, 'clinical': 0, 'genomic': 1},
    'Clinical+Genomic': {'image': 0, 'clinical': 1, 'genomic': 1}
}

ablation_results = {}
for name, config_mask in ablation_configs.items():
    result = evaluate_with_modality_mask(model, test_loader, device, config_mask)
    ablation_results[name] = result
    print(f"   {name:20s}: Acc={result['accuracy']:.4f} | F1={result['f1']:.4f} | AUC={result['auc']:.4f}")

# Save ablation results
ablation_df = pd.DataFrame(ablation_results).T
ablation_df.to_csv('results/ablation_study.csv')
print("\n   � Saved to results/ablation_study.csv")


 Calculating Bootstrap Confidence Intervals...


Bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]


 95% CONFIDENCE INTERVALS
   ACCURACY    : 0.9708 (0.9591 - 0.9805)
   F1          : 0.9712 (0.9599 - 0.9809)
   AUC         : 0.9931 (0.9881 - 0.9970)

 ABLATION STUDY
   All Modalities      : Acc=0.9708 | F1=0.9714 | AUC=0.9926
   Image Only          : Acc=0.9708 | F1=0.9714 | AUC=0.9932
   Clinical Only       : Acc=0.4922 | F1=0.3113 | AUC=0.5082
   Genomic Only        : Acc=0.5895 | F1=0.5567 | AUC=0.6279
   Image+Clinical      : Acc=0.9708 | F1=0.9714 | AUC=0.9938
   Image+Genomic       : Acc=0.9708 | F1=0.9714 | AUC=0.9926
   Clinical+Genomic    : Acc=0.5837 | F1=0.5418 | AUC=0.6268

   � Saved to results/ablation_study.csv


In [ ]:
#@title 16.  Generate Publication Figures (300 DPI)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = config.COLORS

# === Figure 1: ROC Curve ===
fig, ax = plt.subplots(figsize=(8, 7))
fpr, tpr, _ = roc_curve(pred_data['labels'], pred_data['probabilities'][:, 1])
roc_auc = auc(fpr, tpr)

ax.plot(fpr, tpr, color=COLORS['fusion'], lw=2.5, label=f'Multi-Modal Fusion (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.2, color=COLORS['fusion'])

ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve - Multi-Modal Cervical Cancer Classification', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.savefig('figures/fig1_roc.png', dpi=300, bbox_inches='tight')
plt.show()
print(" Figure 1: ROC Curve")

# === Figure 2: Confusion Matrix ===
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm = confusion_matrix(pred_data['labels'], pred_data['predictions'])

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Abnormal'], yticklabels=['Normal', 'Abnormal'],
            annot_kws={'size': 16, 'weight': 'bold'})
axes[0].set_xlabel('Predicted', fontweight='bold')
axes[0].set_ylabel('Actual', fontweight='bold')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Abnormal'], yticklabels=['Normal', 'Abnormal'],
            annot_kws={'size': 16, 'weight': 'bold'})
axes[1].set_xlabel('Predicted', fontweight='bold')
axes[1].set_ylabel('Actual', fontweight='bold')
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fig2_confusion.png', dpi=300, bbox_inches='tight')
plt.show()
print(" Figure 2: Confusion Matrix")

# === Figure 3: Training Curves ===
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
epochs = range(1, len(tracker.get('train_loss')) + 1)

axes[0, 0].plot(epochs, tracker.get('train_loss'), 'b-o', label='Train', markersize=4)
axes[0, 0].plot(epochs, tracker.get('val_loss'), 'r-s', label='Validation', markersize=4)
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Loss', fontweight='bold'); axes[0, 0].legend()

axes[0, 1].plot(epochs, tracker.get('train_acc'), 'b-o', label='Train', markersize=4)
axes[0, 1].plot(epochs, tracker.get('val_acc'), 'r-s', label='Validation', markersize=4)
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy', fontweight='bold'); axes[0, 1].legend()

axes[1, 0].plot(epochs, tracker.get('val_auc'), 'g-D', markersize=4)
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('AUC')
axes[1, 0].set_title('Validation AUC', fontweight='bold')

axes[1, 1].plot(epochs, tracker.get('lr'), color='orange', lw=2)
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule', fontweight='bold')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig('figures/fig3_training.png', dpi=300, bbox_inches='tight')
plt.show()
print(" Figure 3: Training Curves")

# === Figure 4: Ablation Study ===
fig, ax = plt.subplots(figsize=(10, 6))
configs = list(ablation_results.keys())
aucs = [ablation_results[c]['auc'] for c in configs]

colors = plt.cm.RdYlGn(plt.Normalize(min(aucs) - 0.1, max(aucs))(aucs))
bars = ax.barh(configs, aucs, color=colors, edgecolor='black', linewidth=0.5)

for i, (bar, val) in enumerate(zip(bars, aucs)):
    ax.text(val + 0.01, i, f'{val:.3f}', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('AUC-ROC', fontsize=12, fontweight='bold')
ax.set_title('Ablation Study: Modality Contribution', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.1)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Random')

plt.tight_layout()
plt.savefig('figures/fig4_ablation.png', dpi=300, bbox_inches='tight')
plt.show()
print(" Figure 4: Ablation Study")

print("\n� All figures saved to figures/ directory!")

 Figure 1: ROC Curve


 Figure 2: Confusion Matrix


 Figure 3: Training Curves


 Figure 4: Ablation Study

� All figures saved to figures/ directory!


In [ ]:
#@title 17. � Export Results & Download

# Save metrics
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1', 'AUC'],
    'Value': [final_metrics['accuracy'], final_metrics['f1'], final_metrics['auc']],
    'CI_Low': [ci_results['accuracy']['ci_low'], ci_results['f1']['ci_low'], ci_results['auc']['ci_low']],
    'CI_High': [ci_results['accuracy']['ci_high'], ci_results['f1']['ci_high'], ci_results['auc']['ci_high']]
})
metrics_df.to_csv('results/metrics.csv', index=False)

# Save training history
pd.DataFrame(tracker.history).to_csv('results/training_history.csv', index=False)

# Save experiment summary
summary = {
    'experiment': config.EXPERIMENT_NAME,
    'total_images': len(all_paths),
    'sipakmed_images': len(sipakmed_paths),
    'herlev_images': len(herlev_paths),
    'clinical_features': CLINICAL_DIM,
    'genomic_features': GENOMIC_DIM,
    'results': {k: float(v) for k, v in final_metrics.items()},
    'confidence_intervals': ci_results,
    'ablation_study': ablation_results
}

with open('results/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Create ZIP archives
shutil.make_archive('cervical_cancer_results', 'zip', 'results')
shutil.make_archive('cervical_cancer_figures', 'zip', 'figures')
shutil.make_archive('cervical_cancer_models', 'zip', 'models')

print("� Archives created!")

# Download in Colab
try:
    from google.colab import files
    files.download('cervical_cancer_results.zip')
    files.download('cervical_cancer_figures.zip')
    files.download('cervical_cancer_models.zip')
    print(" Downloads started!")
except:
    print(" Files saved locally (not in Colab)")

print("\n" + "="*70)
print(" EXPERIMENT COMPLETE!")
print("="*70)
print(f"\n DATASET SUMMARY:")
print(f"   Total images: {len(all_paths)}")
print(f"   - SIPaKMeD: {len(sipakmed_paths)}")
print(f"   - Herlev: {len(herlev_paths)}")
print(f"\n FINAL PERFORMANCE:")
print(f"   Accuracy: {final_metrics['accuracy']:.4f} ({ci_results['accuracy']['ci_low']:.4f} - {ci_results['accuracy']['ci_high']:.4f})")
print(f"   F1-Score: {final_metrics['f1']:.4f} ({ci_results['f1']['ci_low']:.4f} - {ci_results['f1']['ci_high']:.4f})")
print(f"   AUC-ROC:  {final_metrics['auc']:.4f} ({ci_results['auc']['ci_low']:.4f} - {ci_results['auc']['ci_high']:.4f})")
print("\n" + "="*70)

� Archives created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Downloads started!

 EXPERIMENT COMPLETE!

 DATASET SUMMARY:
   Total images: 6849
   - SIPaKMeD: 5015
   - Herlev: 1834

 FINAL PERFORMANCE:
   Accuracy: 0.9708 (0.9591 - 0.9805)
   F1-Score: 0.9713 (0.9599 - 0.9809)
   AUC-ROC:  0.9931 (0.9881 - 0.9970)



---
#  REVISION ANALYSES : Round 2 (run after Cells 1–15)
Every cell prints the numbers needed for the paper and the response letter. Run **in order**.

| Cell | Addresses | Produces |
|---|---|---|
| R1 | R1.3, R2.13/16/17 | Percentile-bootstrap 95% CIs for ALL metrics + Brier + ECE (Table 5) |
| R2 | R1.2g / R1-1 (2nd rnd) | Per-dataset performance (within-split subsets) |
| R3 | R2.11 + R2-20 (2nd rnd) | T-sensitivity sweep + practical throughput |
| R4 | R2-16 (2nd rnd) | McNemar with EXACT p-values (b, c counts) |
| R5 | R1-4 (2nd rnd) | Static equal-weight fusion baseline + McNemar vs adaptive |
| R6 | R1-1 (2nd rnd) | Cross-dataset transfer: SIPaKMeD↔Herlev |
| R7 | R2-13 (2nd rnd) | Regenerated 300-DPI figures from THIS run |
| R8 | R1.3 | OPTIONAL five-seed mean ± SD (off by default) |
| R9 | R1-3, R2-11/12 (2nd rnd) | CONSOLIDATED summary → revision2_results.json + zip |


In [ ]:
#@title R1. Percentile-bootstrap 95% CIs : ALL metrics + Brier + ECE  [Table 5]
from sklearn.metrics import brier_score_loss, precision_score, recall_score

def compute_ece(y_true, conf, correct, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0: continue
        ece += (m.sum()/len(conf)) * abs(correct[m].mean() - conf[m].mean())
    return ece

y  = pred_data['labels'].astype(int)
yp = pred_data['predictions'].astype(int)
P  = pred_data['probabilities']
conf_top = P.max(1); correct = (yp == y).astype(float)
brier = brier_score_loss(y, P[:, 1])
ece   = compute_ece(y, conf_top, correct, n_bins=15)

def full_bootstrap(y, yp, P, n_boot=1000, ci=0.95):
    n=len(y); rng=np.random.RandomState(42)
    acc,sens,spec,prec,f1s,aucs,briers,eces=[],[],[],[],[],[],[],[]
    for _ in range(n_boot):
        idx=rng.choice(n,n,replace=True)
        if len(np.unique(y[idx]))<2: continue
        yt,ypp,Pp=y[idx],yp[idx],P[idx]
        acc.append(accuracy_score(yt,ypp)); f1s.append(f1_score(yt,ypp,zero_division=0))
        prec.append(precision_score(yt,ypp,zero_division=0)); sens.append(recall_score(yt,ypp,zero_division=0))
        tn=((yt==0)&(ypp==0)).sum(); fp=((yt==0)&(ypp==1)).sum()
        spec.append(tn/(tn+fp) if (tn+fp)>0 else 0)
        aucs.append(roc_auc_score(yt,Pp[:,1])); briers.append(brier_score_loss(yt,Pp[:,1]))
        ct=Pp.max(1); cr=(ypp==yt).astype(float); eces.append(compute_ece(yt,ct,cr))
    def ci_(v):
        v=np.array(v); a=(1-ci)/2
        return v.mean(), np.percentile(v,a*100), np.percentile(v,(1-a)*100)
    return {'Accuracy':ci_(acc),'Sensitivity':ci_(sens),'Specificity':ci_(spec),
            'Precision':ci_(prec),'F1':ci_(f1s),'AUC':ci_(aucs),'Brier':ci_(briers),'ECE':ci_(eces)}

print("="*68)
print("R1.  PERCENTILE-BOOTSTRAP 95% CIs (1,000 resamples of the test set)")
print("     NOTE for paper (R2-17): intervals are PERCENTILE bootstrap;")
print("     they reflect test-set resampling, NOT independent retraining (R1-2).")
print("="*68)
print(f"{'Metric':12s} {'Point':>9s} {'Mean':>9s} {'95% CI Low':>11s} {'95% CI High':>12s}")
cis=full_bootstrap(y,yp,P)
tn0=((y==0)&(yp==0)).sum(); fp0=((y==0)&(yp==1)).sum()
point={'Accuracy':final_metrics['accuracy'],'Sensitivity':final_metrics['recall'],
       'Specificity': tn0/max(tn0+fp0,1),
       'Precision':final_metrics['precision'],'F1':final_metrics['f1'],
       'AUC':final_metrics['auc'],'Brier':brier,'ECE':ece}
for k,(m,lo,hi) in cis.items():
    print(f"{k:12s} {point[k]:9.4f} {m:9.4f} {lo:11.4f} {hi:12.4f}")
ci_full = cis


R1.  PERCENTILE-BOOTSTRAP 95% CIs (1,000 resamples of the test set)
     NOTE for paper (R2-17): intervals are PERCENTILE bootstrap;
     they reflect test-set resampling, NOT independent retraining (R1-2).
Metric           Point      Mean  95% CI Low  95% CI High
Accuracy        0.9708    0.9709      0.9601       0.9805
Sensitivity     0.9807    0.9808      0.9691       0.9907
Specificity     0.9608    0.9609      0.9429       0.9773
Precision       0.9621    0.9622      0.9450       0.9782
F1              0.9713    0.9714      0.9605       0.9809
AUC             0.9931    0.9931      0.9878       0.9970
Brier           0.0246    0.0246      0.0170       0.0333
ECE             0.0138    0.0178      0.0101       0.0268


In [ ]:
#@title R2. Per-dataset performance (within-split subsets : NOT cross-dataset)
from sklearn.metrics import recall_score
def metrics_subset(name, sel):
    yt=y[sel]; ypp=yp[sel]; Pp=P[sel]
    if len(np.unique(yt))<2:
        print(f"   {name}: single class, skipped"); return None
    tn=((yt==0)&(ypp==0)).sum(); fp=((yt==0)&(ypp==1)).sum()
    spec=tn/(tn+fp) if (tn+fp)>0 else float('nan')
    rng=np.random.RandomState(42); aucs=[]
    for _ in range(1000):
        bi=rng.choice(len(yt),len(yt),replace=True)
        if len(np.unique(yt[bi]))<2: continue
        aucs.append(roc_auc_score(yt[bi],Pp[bi,1]))
    lo,hi=np.percentile(aucs,2.5),np.percentile(aucs,97.5)
    r={'n':int(len(yt)),'AUC':float(roc_auc_score(yt,Pp[:,1])),'AUC_lo':float(lo),'AUC_hi':float(hi),
       'Acc':float(accuracy_score(yt,ypp)),'Sens':float(recall_score(yt,ypp,zero_division=0)),
       'Spec':float(spec),'F1':float(f1_score(yt,ypp,zero_division=0))}
    print(f"   {name:10s} n={r['n']:4d} | AUC={r['AUC']:.4f} ({lo:.3f}-{hi:.3f}) | "
          f"Acc={r['Acc']*100:5.2f}% | Sens={r['Sens']*100:5.2f}% | Spec={r['Spec']*100:5.2f}% | F1={r['F1']*100:5.2f}%")
    return r
print("="*68); print("R2.  PER-DATASET (subsets of the pooled held-out split)"); print("="*68)
per={}
for name,key in [('SIPaKMeD','sipakmed'),('Herlev','herlev')]:
    sel=np.where(test_source==key)[0]
    if len(sel)>0: per[name]=metrics_subset(name, sel)
print(f"\n   Pooled     n={len(y):4d} | AUC={final_metrics['auc']:.4f} | Acc={final_metrics['accuracy']*100:5.2f}%")
print("\n   \u26A0 Manuscript wording (R1-1): these are per-dataset SUBSETS of the")
print("     pooled split \u2014 do NOT describe them as cross-dataset/external validation.")


R2.  PER-DATASET (subsets of the pooled held-out split)
   SIPaKMeD   n= 735 | AUC=0.9964 (0.993-0.999) | Acc=98.10% | Sens=97.37% | Spec=98.61% | F1=97.69%
   Herlev     n= 293 | AUC=0.9697 (0.939-0.990) | Acc=94.54% | Sens=99.07% | Spec=82.28% | F1=96.36%

   Pooled     n=1028 | AUC=0.9931 | Acc=97.08%

    Manuscript wording (R1-1): these are per-dataset SUBSETS of the
     pooled split : do NOT describe them as cross-dataset/external validation.


In [ ]:
#@title R3. T-sensitivity sweep + practical throughput  [R2.11, R2-20]
import time
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
def eval_with_T(T):
    for mc in [model.image_mc, model.clinical_mc, model.genomic_mc]:
        mc.n_samples = T
    model.eval(); labels_,probs_=[],[]
    t0=time.time(); n_seen=0
    with torch.no_grad():
        for batch in test_loader:
            img=batch['image'].to(device); clin=batch['clinical'].to(device)
            gen=batch['genomic'].to(device); mask=batch['modality_mask'].to(device)
            out=model(img,clin,gen,mask,return_uncertainty=True,return_details=True)
            probs_.extend(out['prediction'].cpu().numpy()); labels_.extend(batch['label'].numpy())
            n_seen+=img.size(0)
    dt=(time.time()-t0)/max(n_seen,1)
    yt=np.array(labels_); Pp=np.array(probs_); ypp=Pp.argmax(1)
    ct=Pp.max(1); cr=(ypp==yt).astype(float)
    return {'AUC':float(roc_auc_score(yt,Pp[:,1])),'Acc':float(accuracy_score(yt,ypp)),
            'ECE':float(compute_ece(yt,ct,cr)),'Brier':float(brier_score_loss(yt,Pp[:,1])),'t':float(dt)}
print("="*68); print(f"R3.  T-SENSITIVITY on {GPU_NAME}"); print("="*68)
print(f"{'T':>4s} {'AUC':>8s} {'Accuracy':>9s} {'ECE':>7s} {'Brier':>7s} {'s/sample':>9s} {'samples/min':>12s}")
T_rows={}
for T in [5,10,20,30,50]:
    r=eval_with_T(T); r['throughput_per_min']=60.0/r['t']; T_rows[T]=r
    print(f"{T:4d} {r['AUC']:8.4f} {r['Acc']*100:8.2f}% {r['ECE']:7.4f} {r['Brier']:7.4f} "
          f"{r['t']:9.4f} {r['throughput_per_min']:12.0f}")
for mc in [model.image_mc, model.clinical_mc, model.genomic_mc]: mc.n_samples=20
accs=[T_rows[T]['Acc']*100 for T in T_rows]; aucs=[T_rows[T]['AUC'] for T in T_rows]
print(f"\n   Accuracy spread across T: {max(accs)-min(accs):.2f} pp | AUC spread: {max(aucs)-min(aucs):.4f}")
print(f"   PAPER (R2-20): at T=20, {GPU_NAME} processes \u2248{T_rows[20]['throughput_per_min']:.0f} samples/min "
      f"(\u2248{T_rows[20]['throughput_per_min']*60:.0f}/hour).")


R3.  T-SENSITIVITY on Tesla T4
   T      AUC  Accuracy     ECE   Brier  s/sample  samples/min
   5   0.9931    97.08%  0.0162  0.0247    0.0339         1771
  10   0.9930    97.18%  0.0160  0.0250    0.0686          875
  20   0.9931    97.08%  0.0136  0.0245    0.1386          433
  30   0.9931    96.98%  0.0158  0.0245    0.2099          286
  50   0.9931    97.08%  0.0169  0.0247    0.3487          172

   Accuracy spread across T: 0.19 pp | AUC spread: 0.0001
   PAPER (R2-20): at T=20, Tesla T4 processes ≈433 samples/min (≈25978/hour).


In [ ]:
#@title R4. Masked-configuration probabilities + McNemar EXACT p-values  [R2-16]
try:
    from statsmodels.stats.contingency_tables import mcnemar as _mcnemar
    def mcnemar_exact(b,c):
        return float(_mcnemar([[0,b],[c,0]], exact=True).pvalue)
except Exception:
    from scipy.stats import binomtest
    def mcnemar_exact(b,c):
        n=b+c
        return 1.0 if n==0 else float(binomtest(min(b,c), n, 0.5).pvalue)

def probs_for_mask(mi, mc_, mg):
    fixed=torch.tensor([mi,mc_,mg]).float()
    model.eval(); out=[]
    with torch.no_grad():
        for batch in test_loader:
            img=batch['image'].to(device); clin=batch['clinical'].to(device)
            gen=batch['genomic'].to(device)
            m=fixed.unsqueeze(0).expand(img.size(0),-1).to(device)
            pr,_=model(img,clin,gen,m)
            out.extend(pr.cpu().numpy())
    return np.array(out)

configs={'Image Only':(1,0,0),'Clinical Only':(0,1,0),'Genomic Only':(0,0,1),
         'Image+Clinical':(1,1,0),'Image+Genomic':(1,0,1),
         'Clinical+Genomic':(0,1,1),'All Modalities':(1,1,1)}
probs_cfg={}; preds_cfg={}
for name,msk in configs.items():
    probs_cfg[name]=probs_for_mask(*msk); preds_cfg[name]=probs_cfg[name].argmax(1)
    print(f"   evaluated {name:18s} AUC={roc_auc_score(y,probs_cfg[name][:,1]):.4f}")

print()
print("="*68); print("R4.  McNEMAR EXACT TESTS vs Image-Only  (report these p-values)"); print("="*68)
mcnemar_out={}
base_pred=preds_cfg['Image Only']
for name in ['Image+Clinical','Image+Genomic','All Modalities']:
    other=preds_cfg[name]
    b=int(((base_pred==y)&(other!=y)).sum()); c=int(((base_pred!=y)&(other==y)).sum())
    p=mcnemar_exact(b,c); mcnemar_out['ImageOnly_vs_'+name]={'b':b,'c':c,'p':p}
    verdict='n.s.' if p>0.05 else 'SIGNIFICANT'
    print(f"   Image-Only vs {name:16s}: b={b:3d}, c={c:3d}, exact p = {p:.4f} ({verdict})")


   evaluated Image Only         AUC=0.9932
   evaluated Clinical Only      AUC=0.5082
   evaluated Genomic Only       AUC=0.6279
   evaluated Image+Clinical     AUC=0.9938
   evaluated Image+Genomic      AUC=0.9926
   evaluated Clinical+Genomic   AUC=0.6268
   evaluated All Modalities     AUC=0.9926

R4.  McNEMAR EXACT TESTS vs Image-Only  (report these p-values)
   Image-Only vs Image+Clinical  : b=  0, c=  0, exact p = 1.0000 (n.s.)
   Image-Only vs Image+Genomic   : b=  0, c=  0, exact p = 1.0000 (n.s.)
   Image-Only vs All Modalities  : b=  0, c=  0, exact p = 1.0000 (n.s.)


In [ ]:
#@title R5. Static equal-weight fusion baseline + McNemar vs adaptive  [R1-4]
P_static=(probs_cfg['Image Only']+probs_cfg['Clinical Only']+probs_cfg['Genomic Only'])/3.0
yp_static=P_static.argmax(1)
tn=((y==0)&(yp_static==0)).sum(); fp=((y==0)&(yp_static==1)).sum()
static_res={'AUC':float(roc_auc_score(y,P_static[:,1])),
            'Acc':float(accuracy_score(y,yp_static)),
            'F1':float(f1_score(y,yp_static,zero_division=0)),
            'Spec':float(tn/(tn+fp)) if (tn+fp)>0 else float('nan')}
print("="*68); print("R5.  STATIC EQUAL-WEIGHT FUSION (w = 1/3 each; add as ablation row)"); print("="*68)
print(f"   Static fusion : AUC={static_res['AUC']:.4f}  Acc={static_res['Acc']*100:.2f}%  F1={static_res['F1']*100:.2f}%")
ya=preds_cfg['All Modalities']
print(f"   Adaptive (All): AUC={roc_auc_score(y,probs_cfg['All Modalities'][:,1]):.4f}  "
      f"Acc={accuracy_score(y,ya)*100:.2f}%")
b=int(((ya==y)&(yp_static!=y)).sum()); c=int(((ya!=y)&(yp_static==y)).sum())
p=mcnemar_exact(b,c); static_res['mcnemar_vs_adaptive']={'b':b,'c':c,'p':p}
if p<=0.05 and b>c: verdict='adaptive significantly better'
elif p>0.05: verdict='n.s.'
else: verdict='SIGNIFICANT'
print(f"   McNemar adaptive vs static: b={b}, c={c}, exact p = {p:.4f} ({verdict})")
print()
print("   Expected: static fusion is dragged down by uninformative modalities;")
print("   adaptive fusion preserves image-level performance. Report whatever prints.")


R5.  STATIC EQUAL-WEIGHT FUSION (w = 1/3 each; add as ablation row)
   Static fusion : AUC=0.9824  Acc=97.28%  F1=97.31%
   Adaptive (All): AUC=0.9926  Acc=97.08%
   McNemar adaptive vs static: b=3, c=5, exact p = 0.7266 (n.s.)

   Expected: static fusion is dragged down by uninformative modalities;
   adaptive fusion preserves image-level performance. Report whatever prints.


In [23]:
#@title R6. Cross-dataset transfer: SIPaKMeD <-> Herlev  [R1-1]  (~2 subset trainings)
from sklearn.model_selection import train_test_split as tts
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler
from sklearn.metrics import recall_score

cross_results={}
for src_name, tgt_name in [('sipakmed','herlev'), ('herlev','sipakmed')]:
    print(f"\n===== TRAIN on {src_name.upper()}  ->  TEST on {tgt_name.upper()} =====")
    set_seed(42)
    src=np.where(all_source==src_name)[0]; tgt=np.where(all_source==tgt_name)[0]
    tr,va=tts(src,test_size=0.15,stratify=all_labels[src],random_state=42)
    ds_tr=MultiModalDataset([all_paths[i] for i in tr], clinical_X[tr], genomic_X[tr],
                            all_labels[tr], train_transform, config.MODALITY_DROPOUT_PROB, True)
    ds_va=MultiModalDataset([all_paths[i] for i in va], clinical_X[va], genomic_X[va],
                            all_labels[va], val_transform, 0.0, False)
    ds_xt=MultiModalDataset([all_paths[i] for i in tgt], clinical_X[tgt], genomic_X[tgt],
                            all_labels[tgt], val_transform, 0.0, False)
    tl=DataLoader(ds_tr,batch_size=config.BATCH_SIZE,shuffle=True,num_workers=config.NUM_WORKERS,pin_memory=True)
    vl=DataLoader(ds_va,batch_size=config.BATCH_SIZE,shuffle=False,num_workers=config.NUM_WORKERS,pin_memory=True)
    xl=DataLoader(ds_xt,batch_size=config.BATCH_SIZE,shuffle=False,num_workers=config.NUM_WORKERS,pin_memory=True)
    m=MultiModalModel(CLINICAL_DIM,GENOMIC_DIM,config.NUM_CLASSES,
                      config.DROPOUT_RATE,config.MC_DROPOUT_SAMPLES).to(device)
    opt=optim.AdamW(m.parameters(),lr=config.LEARNING_RATE,weight_decay=config.WEIGHT_DECAY)
    sch=OneCycleLR(opt,max_lr=config.LEARNING_RATE,total_steps=len(tl)*config.NUM_EPOCHS,pct_start=0.1)
    scaler=GradScaler(enabled=config.USE_AMP)
    es=EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE); crit=FocalLoss()
    for ep in range(config.NUM_EPOCHS):
        train_epoch(m,tl,crit,opt,sch,scaler,device)
        vm=evaluate(m,vl,crit,device); es(vm['auc'],m)
        if es.early_stop:
            print(f"  early stop @ epoch {ep+1}"); break
    m.load_state_dict(es.best_state)
    fm,pd_=evaluate(m,xl,crit,device,return_preds=True)
    y_=pd_['labels'].astype(int); yp_=pd_['predictions'].astype(int); P_=pd_['probabilities']
    tn=((y_==0)&(yp_==0)).sum(); fp=((y_==0)&(yp_==1)).sum()
    rng=np.random.RandomState(42); aucs=[]
    for _ in range(1000):
        bi=rng.choice(len(y_),len(y_),replace=True)
        if len(np.unique(y_[bi]))<2: continue
        aucs.append(roc_auc_score(y_[bi],P_[bi,1]))
    lo,hi=np.percentile(aucs,2.5),np.percentile(aucs,97.5)
    r={'n_test':int(len(y_)),'AUC':float(roc_auc_score(y_,P_[:,1])),'AUC_lo':float(lo),'AUC_hi':float(hi),
       'Acc':float(accuracy_score(y_,yp_)),'Sens':float(recall_score(y_,yp_,zero_division=0)),
       'Spec':float(tn/(tn+fp)) if (tn+fp)>0 else float('nan'),'F1':float(f1_score(y_,yp_,zero_division=0))}
    cross_results[src_name+'->'+tgt_name]=r
    print(f"  RESULT n={r['n_test']}  AUC={r['AUC']:.4f} ({lo:.3f}-{hi:.3f})  Acc={r['Acc']*100:.2f}%  "
          f"Sens={r['Sens']*100:.2f}%  Spec={r['Spec']*100:.2f}%  F1={r['F1']*100:.2f}%")
set_seed(42)
print()
print("   Report BOTH directions honestly - a drop vs pooled results is the")
print("   expected finding and motivates the external-validation limitation.")



===== TRAIN on SIPAKMED  ->  TEST on HERLEV =====


Training:   0%|          | 0/134 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

Training:   0%|          | 0/134 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

...

Training:   0%|          | 0/49 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

  early stop @ epoch 22


Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

  RESULT n=5015  AUC=0.7470 (0.733-0.761)  Acc=68.61%  Sens=59.93%  Spec=74.86%  F1=61.52%

   Report BOTH directions honestly - a drop vs pooled results is the
   expected finding and motivates the external-validation limitation.


In [24]:
#@title R7. Regenerate 300-DPI figures from THIS run  [R2-13 consistency]
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, confusion_matrix
import seaborn as sns
os.makedirs('figures', exist_ok=True)

names=list(configs.keys())+['Static Equal-Weight']
aucs_fig=[roc_auc_score(y,probs_cfg[n][:,1]) for n in configs]+[static_res['AUC']]
plt.figure(figsize=(9,5))
cols=['#1F4E79' if n=='All Modalities' else '#C00000' if 'Static' in n
      else '#2E75B6' if 'Image' in n else '#7F7F7F' for n in names]
plt.bar(range(len(names)),aucs_fig,color=cols)
for i,v in enumerate(aucs_fig): plt.text(i,v+0.01,f"{v:.3f}",ha='center',fontsize=9)
plt.xticks(range(len(names)),names,rotation=30,ha='right'); plt.ylabel('AUC-ROC')
plt.ylim(0.4,1.08); plt.title('Ablation: modality configurations (current run)')
plt.tight_layout(); plt.savefig('figures/fig_ablation.png',dpi=300,bbox_inches='tight'); plt.show()

edges=np.linspace(0,1,16); ctr=(edges[:-1]+edges[1:])/2
accb,dens=[],[]
for lo,hi in zip(edges[:-1],edges[1:]):
    mm=(conf_top>lo)&(conf_top<=hi); dens.append(mm.mean())
    accb.append(correct[mm].mean() if mm.sum() else np.nan)
fig,ax=plt.subplots(figsize=(7,5.5)); ax2=ax.twinx()
ax2.bar(ctr,dens,width=0.06,color='#BBD3E8',alpha=0.55,label='Prediction density')
ax2.set_ylabel('Prediction density')
ax.plot([0,1],[0,1],'k--',lw=1.5,label='Perfect calibration')
ax.plot(ctr,accb,'o-',color='#C0392B',lw=2,label=f'Model (ECE = {ece:.3f})')
ax.set_xlabel('Confidence'); ax.set_ylabel('Empirical accuracy')
ax.set_title(f'Reliability Diagram (ECE = {ece:.3f}, Brier = {brier:.3f})')
h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
ax.legend(h1+h2,l1+l2,loc='upper left')
plt.tight_layout(); plt.savefig('figures/fig_reliability.png',dpi=300,bbox_inches='tight'); plt.show()

fpr,tpr,_=roc_curve(y,P[:,1])
plt.figure(figsize=(6,5.5))
plt.plot(fpr,tpr,color='#1F4E79',lw=2,label=f"AUC = {final_metrics['auc']:.3f}")
plt.plot([0,1],[0,1],'k--',lw=1); plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('ROC curve (held-out test set)'); plt.legend(loc='lower right')
plt.tight_layout(); plt.savefig('figures/fig_roc.png',dpi=300,bbox_inches='tight'); plt.show()

cm=confusion_matrix(y,yp)
plt.figure(figsize=(5.5,4.6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',cbar=False,
            xticklabels=['Normal','Abnormal'],yticklabels=['Normal','Abnormal'])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (n = %d)'%len(y))
plt.tight_layout(); plt.savefig('figures/fig_confusion.png',dpi=300,bbox_inches='tight'); plt.show()
print("Saved figures/fig_ablation.png, fig_reliability.png, fig_roc.png, fig_confusion.png (300 DPI)")
print("Replace the OLD manuscript figures with these so figures and tables match (R2-13).")


Saved figures/fig_ablation.png, fig_reliability.png, fig_roc.png, fig_confusion.png (300 DPI)
Replace the OLD manuscript figures with these so figures and tables match (R2-13).


In [25]:
#@title R8. OPTIONAL : Five-run mean ± SD (retrains 5x; toggle to run)  [R1.3]
RUN_MULTISEED = False  #@param {type:"boolean"}
if RUN_MULTISEED:
    import torch.optim as optim
    from torch.optim.lr_scheduler import OneCycleLR
    from torch.cuda.amp import GradScaler
    from collections import defaultdict
    seeds=[42,1,2,3,4]; agg=defaultdict(list)
    for s in seeds:
        print(f"\n===== SEED {s} =====")
        set_seed(s)
        m=MultiModalModel(CLINICAL_DIM,GENOMIC_DIM,config.NUM_CLASSES,
                          config.DROPOUT_RATE,config.MC_DROPOUT_SAMPLES).to(device)
        opt=optim.AdamW(m.parameters(),lr=config.LEARNING_RATE,weight_decay=config.WEIGHT_DECAY)
        sch=OneCycleLR(opt,max_lr=config.LEARNING_RATE,total_steps=len(train_loader)*config.NUM_EPOCHS,pct_start=0.1)
        scaler=GradScaler(enabled=config.USE_AMP)
        es=EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE); crit=FocalLoss()
        for ep in range(config.NUM_EPOCHS):
            train_epoch(m,train_loader,crit,opt,sch,scaler,device)
            vm=evaluate(m,val_loader,crit,device); es(vm['auc'],m)
            if es.early_stop: break
        m.load_state_dict(es.best_state)
        fm,pd_=evaluate(m,test_loader,crit,device,return_preds=True)
        yy=pd_['labels'].astype(int); pp=pd_['predictions'].astype(int); PP=pd_['probabilities']
        tn=((yy==0)&(pp==0)).sum(); fp=((yy==0)&(pp==1)).sum()
        agg['Accuracy'].append(fm['accuracy']*100); agg['Sensitivity'].append(fm['recall']*100)
        agg['Specificity'].append(100*tn/max(tn+fp,1)); agg['Precision'].append(fm['precision']*100)
        agg['F1'].append(fm['f1']*100); agg['AUC'].append(fm['auc'])
        agg['Brier'].append(brier_score_loss(yy,PP[:,1]))
        ct=PP.max(1); cr=(pp==yy).astype(float); agg['ECE'].append(compute_ece(yy,ct,cr))
        print(f"  Acc={fm['accuracy']*100:.2f}%  AUC={fm['auc']:.4f}")
    set_seed(42)
    print("\n"+"="*60); print("R8.  FIVE-RUN MEAN +/- SD"); print("="*60)
    five_seed={}
    for k,v in agg.items():
        v=np.array(v); five_seed[k]={'mean':float(v.mean()),'sd':float(v.std())}
        dec=4 if k in ('AUC','Brier','ECE') else 2
        print(f"   {k:12s}: {v.mean():.{dec}f} +/- {v.std():.{dec}f}")
else:
    five_seed=None
    print("RUN_MULTISEED = False - skipped. Bootstrap CIs (R1) remain the reported dispersion;")
    print("if skipped, keep the manuscript statement that CIs reflect resampling, not retraining.")


RUN_MULTISEED = False - skipped. Bootstrap CIs (R1) remain the reported dispersion;
if skipped, keep the manuscript statement that CIs reflect resampling, not retraining.


In [26]:
#@title R9. � CONSOLIDATED ROUND-2 SUMMARY -> revision2_results.json + zip
from sklearn.metrics import confusion_matrix as _cmx
import shutil, json as _json
cm=_cmx(y,yp); tn,fp,fn,tp=cm.ravel()
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("="*70)
print("  FINAL ROUND-2 RESULTS - single source of truth for the paper")
print(f"  GPU: {GPU_NAME}   (use THIS name consistently in the manuscript)")
print("="*70)
print(f"  Test n={len(y)} (Normal={int((y==0).sum())}, Abnormal={int((y==1).sum())})")
print(f"  Confusion: TN={tn} FP={fp} FN={fn} TP={tp}")
print("-"*70); print("  HEADLINE (point | percentile-bootstrap 95% CI):")
for k,(m,lo,hi) in ci_full.items():
    print(f"    {k:12s}: {point[k]:.4f}  [{lo:.4f}, {hi:.4f}]")
print("-"*70); print("  ABLATION (this run) + static baseline:")
for n in configs:
    pr=probs_cfg[n]; ypn=pr.argmax(1)
    print(f"    {n:20s}: AUC={roc_auc_score(y,pr[:,1]):.4f}  Acc={accuracy_score(y,ypn)*100:5.2f}%")
print(f"    {'Static Equal-Weight':20s}: AUC={static_res['AUC']:.4f}  Acc={static_res['Acc']*100:5.2f}%")
print("-"*70); print("  McNEMAR EXACT p-VALUES (R2-16):")
for k,v in mcnemar_out.items(): print(f"    {k}: b={v['b']}, c={v['c']}, p={v['p']:.4f}")
sv=static_res['mcnemar_vs_adaptive']
print(f"    Adaptive_vs_Static: b={sv['b']}, c={sv['c']}, p={sv['p']:.4f}")
print("-"*70); print("  PER-DATASET (subsets):")
for name,r in per.items():
    if r: print(f"    {name:10s}: AUC={r['AUC']:.4f} ({r['AUC_lo']:.3f}-{r['AUC_hi']:.3f}) Acc={r['Acc']*100:.2f}% Spec={r['Spec']*100:.2f}%")
print("-"*70); print("  CROSS-DATASET TRANSFER (R1-1):")
for k,r in cross_results.items():
    print(f"    {k:22s}: n={r['n_test']} AUC={r['AUC']:.4f} ({r['AUC_lo']:.3f}-{r['AUC_hi']:.3f}) Acc={r['Acc']*100:.2f}%")
print("-"*70); print("  T-SENSITIVITY + THROUGHPUT (R2-20):")
for T,r in T_rows.items():
    print(f"    T={T:2d}: AUC={r['AUC']:.4f} ECE={r['ECE']:.4f} {r['t']:.4f}s/sample ~{r['throughput_per_min']:.0f}/min")
print("="*70)
out={'gpu':GPU_NAME,
     'headline_ci':{k:{'point':float(point[k]),'ci_low':float(v[1]),'ci_high':float(v[2])} for k,v in ci_full.items()},
     'confusion':{'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)},
     'ablation':{n:{'auc':float(roc_auc_score(y,probs_cfg[n][:,1])),
                    'acc':float(accuracy_score(y,probs_cfg[n].argmax(1)))} for n in configs},
     'static_fusion':static_res,'mcnemar_exact':mcnemar_out,
     'per_dataset':per,'cross_dataset':cross_results,
     't_sensitivity':{str(T):{kk:float(vv) for kk,vv in r.items()} for T,r in T_rows.items()},
     'five_seed':five_seed,
     'bootstrap_method':'percentile (1,000 resamples of the held-out test set)'}
os.makedirs('results',exist_ok=True)
with open('results/revision2_results.json','w') as f: _json.dump(out,f,indent=2,default=float)
os.makedirs('revision2_outputs',exist_ok=True)
for src_dir in ['results','figures']:
    if os.path.isdir(src_dir): shutil.copytree(src_dir,'revision2_outputs/'+src_dir,dirs_exist_ok=True)
shutil.make_archive('revision2_package','zip','revision2_outputs')
print("Saved results/revision2_results.json and revision2_package.zip")
try:
    from google.colab import files; files.download('revision2_package.zip')
except Exception: pass


  FINAL ROUND-2 RESULTS - single source of truth for the paper
  GPU: Tesla T4   (use THIS name consistently in the manuscript)
  Test n=1028 (Normal=510, Abnormal=518)
  Confusion: TN=490 FP=20 FN=10 TP=508
----------------------------------------------------------------------
  HEADLINE (point | percentile-bootstrap 95% CI):
    Accuracy    : 0.9708  [0.9601, 0.9805]
    Sensitivity : 0.9807  [0.9691, 0.9907]
    Specificity : 0.9608  [0.9429, 0.9773]
    Precision   : 0.9621  [0.9450, 0.9782]
    F1          : 0.9713  [0.9605, 0.9809]
    AUC         : 0.9931  [0.9878, 0.9970]
    Brier       : 0.0246  [0.0170, 0.0333]
    ECE         : 0.0138  [0.0101, 0.0268]
----------------------------------------------------------------------
  ABLATION (this run) + static baseline:
    Image Only          : AUC=0.9932  Acc=97.08%
    Clinical Only       : AUC=0.5082  Acc=49.22%
    Genomic Only        : AUC=0.6279  Acc=58.95%
    Image+Clinical      : AUC=0.9938  Acc=97.08%
    Image+Genomic  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>